# 05.3 — Gradient clipping

Gradients can spike — a hard batch, a bad-luck initialization, the sharp curvature of a recurrent net — and one huge gradient step can knock a model into the NaN abyss it never returns from ([02.8 §2.2](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). **Gradient clipping** caps the gradient's magnitude before the optimizer step, trading a little bias for a lot of stability. This project clips every step (`gradient_threshold: 100` in `base.yaml`).

This notebook covers:

* Why gradients explode, especially in RNNs.
* Norm clipping: rescaling the whole gradient to a max L2 norm.
* Where clipping goes in the loop (between `backward` and `step`).
* `Global` vs `SubNetwork` clip — the two MATLAB modes.

**Prerequisites:** [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [05.1 the loop](05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

`trainingOptions` exposes clipping directly, and the pipeline sets it:

```matlab
'GradientThreshold', 100, ...
'GradientThresholdMethod', 'l2norm', ...   % clip by the global L2 norm
```

`cfg.GradientClipType` chooses `'Global'` (one norm over all parameters) or `'SubNetwork'` (clip each subnetwork separately). The default is `'Global'` with threshold 100. PyTorch's `clip_grad_norm_` is the direct equivalent — same L2-norm rescaling.

## Section 2 — The Python concepts you need

### 2.1 — Why gradients explode

Recurrent nets multiply gradients through many timesteps in the backward pass; if the factors are >1, the product grows exponentially with sequence length — the classic exploding-gradient problem. The result is an occasional gradient with a huge norm, and a single step from it moves the weights so far the loss diverges. Let's manufacture one:

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
# A pathological gradient: mostly small, but one giant component
grad = torch.randn(1000) * 0.1
grad[0] = 500.0
print(f"gradient L2 norm: {grad.norm():.1f}  — one bad component dominates")
print(f"a step of lr·grad would move weight[0] by {0.01 * grad[0]:.1f} — catastrophic")

### 2.2 — Norm clipping: rescale, don't truncate

The right fix isn't to clip each component (that changes the *direction*) — it's to rescale the *whole* gradient so its L2 norm is at most `max_norm`, preserving direction. If the norm is already under the threshold, nothing happens:

In [ ]:
def clip_norm(g, max_norm):
    norm = g.norm()
    if norm > max_norm:
        return g * (max_norm / norm)     # rescale to exactly max_norm, SAME direction
    return g                              # already fine — untouched

clipped = clip_norm(grad, max_norm=10.0)
print(f"before: norm {grad.norm():.1f}")
print(f"after : norm {clipped.norm():.1f}  (rescaled to the 10.0 cap)")
print(f"direction preserved? {torch.allclose(grad / grad.norm(), clipped / clipped.norm())}")

Direction unchanged, magnitude capped — the model still steps *toward* the same descent direction, just not off a cliff. This is what makes norm clipping safe: it's a scalar rescale of the entire gradient vector, so it never distorts *which way* to move, only *how far*.

### 2.3 — `torch.nn.utils.clip_grad_norm_`

PyTorch's built-in does exactly §2.2 across all of a model's parameters at once — treating them as one concatenated vector. It runs in place on `.grad` (note the trailing underscore, [02.4 §2.5](../02_numpy_and_pytorch_basics/02.4_pytorch_tensors_intro.ipynb)) and returns the *pre-clip* norm (handy for monitoring):

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1))
x, y = torch.randn(4, 10), torch.randn(4, 1)

# Manufacture a large gradient with a big loss scale:
loss = nn.functional.mse_loss(model(x), y) * 1e4
loss.backward()

total_norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
total_norm_after = sum(p.grad.norm()**2 for p in model.parameters())**0.5
print(f"norm before clip: {total_norm_before:.2f}  (the returned value)")
print(f"norm after clip:  {total_norm_after:.2f}  (capped at max_norm=1.0)")

### 2.4 — Where clipping goes in the loop

Clipping happens in the one window where gradients exist but aren't applied yet: **after `backward()`, before `step()`** (and, with accumulation, after the *last* micro-batch's backward, so it clips the full accumulated gradient):

In [ ]:
def train_step(model, opt, x, y, clip):
    opt.zero_grad()
    loss = nn.functional.mse_loss(model(x), y)
    loss.backward()                                           # gradients now exist
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)  # ← clip HERE
    opt.step()                                                 # apply the clipped gradient
    return loss.item()

torch.manual_seed(0)
m = nn.Sequential(nn.Linear(10, 1))
opt = torch.optim.SGD(m.parameters(), lr=0.1)
print("one clipped step, loss:", round(train_step(m, opt, x[:, :10] if x.shape[1]>=10 else torch.randn(4,10), y, clip=1.0), 4))

### 2.5 — Global vs SubNetwork

The two `GradientClipType` modes differ in *what* gets treated as one vector:

| Mode | Clips over | Effect |
|---|---|---|
| **Global** (default) | ALL parameters as one concatenated vector | one norm, one rescale — the encoder and classifier share the budget |
| **SubNetwork** | each submodule separately | encoder clipped independently of classifier — one subnetwork's spike doesn't shrink another's gradient |

Global is simpler and the production default; SubNetwork matters when subnetworks have very different gradient scales (e.g. a huge encoder + tiny classifier) and you don't want one to throttle the other. The project's composite is balanced enough that Global suffices — the config sets `gradient_clip_type: Global`.

## Section 3 — The `neural_data_decoding` implementation

Clipping in `train_one_epoch` — one line, in the right place:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/loop.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "clip_grad_norm_" in line)
for k in range(max(0, i - 4), min(i + 4, len(src))):
    marker = " ►" if "clip_grad_norm_" in src[k] else "  "
    print(f"{k + 1:4}{marker}| {src[k]}")

Things to spot:

* `nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)` — the Global clip (§2.5), over all composite parameters at once.
* It sits after the backward (or accumulation loop) and before `optimizer.step()` — §2.4's window.
* Guarded by `if grad_clip_norm is not None` — `None` disables clipping (a config with no threshold trains unclipped). `grad_clip_norm` comes from `cfg.gradient_threshold`, defaulting to 100.
* The composite is one `nn.Module`, so `model.parameters()` spans encoder + bottleneck + decoder + classifier — Global clip is the natural default for a single composite.

## Section 4 — Hands-on exercises

### Exercise 4.1 — clipping preserves direction

Take a random 50-d gradient, clip it to various thresholds, and confirm the direction (unit vector) is unchanged whenever clipping actually fires.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
g = torch.randn(50) * 5
unit_before = g / g.norm()
for thr in [0.5, 2.0, g.norm().item() + 1]:
    clipped = clip_norm(g, thr)
    fired = clipped.norm() < g.norm() - 1e-6
    same_dir = torch.allclose(unit_before, clipped / clipped.norm(), atol=1e-6)
    print(f"  threshold {thr:6.2f}: clipped={fired}, direction preserved={same_dir}")

### Exercise 4.2 — clip vs no-clip on an exploding step

Take the large-loss setup from §2.3 and show that WITHOUT clipping, a single SGD step sends a weight far from its start; WITH clipping, it moves a controlled amount.

In [ ]:
# Reveal:
def one_step(clip):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(10, 100), nn.ReLU(), nn.Linear(100, 1))
    w0 = m[0].weight[0, 0].item()
    opt = torch.optim.SGD(m.parameters(), lr=0.1)
    (nn.functional.mse_loss(m(torch.randn(4, 10)), torch.randn(4, 1)) * 1e4).backward()
    if clip: torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    opt.step()
    return abs(m[0].weight[0, 0].item() - w0)

print(f"weight moved WITHOUT clip: {one_step(False):8.2f}")
print(f"weight moved WITH    clip: {one_step(True):8.2f}  (bounded)")

## Section 5 — Common errors

### Loss suddenly becomes NaN/inf mid-training

An exploding gradient took a catastrophic step ([02.8 §5](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). Clip the gradient (this project always does); if it still happens, the loss itself may produce infs (log(0), etc.).

### Clipping placed wrong

After `step()` it's useless (gradient already applied); before `backward()` there's nothing to clip. It goes strictly between — and with accumulation, after the *final* micro-batch backward ([05.2](05.2_gradient_accumulation.ipynb)).

### Training stalls after adding clipping

Threshold too low — you're clipping *every* step, throttling legitimate large gradients early in training. Raise the threshold (100 is the project default); clipping should be a safety net, not a constant brake.

### `clip_grad_norm_` return value ignored

It returns the *pre-clip* norm — log it. A gradient norm that regularly exceeds your threshold is diagnostic (learning rate too high, loss mis-scaled).

### Per-component clip expected but Global used (or vice versa)

`Global` shares one budget across all parameters; `SubNetwork` clips each separately (§2.5). If one subnetwork's spikes are shrinking another's gradients unexpectedly, that's Global doing its job — switch to SubNetwork if that's not what you want.

## Section 6 — Further reading

- [clip_grad_norm_ docs](https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html) — the exact rescaling and the return value.
- [On the difficulty of training RNNs (Pascanu et al.)](https://arxiv.org/abs/1211.5063) — the exploding-gradient analysis that motivated norm clipping.
- [`src/neural_data_decoding/training/loop.py`](../../src/neural_data_decoding/training/loop.py) — clipping in the training step.

Next up: **[05.4 learning rate scheduling](05.4_learning_rate_scheduling.ipynb)** — decaying and warming the learning rate over training.